In [3]:
import pandas as pd
import nflreadpy as nfl
from collections import Counter
print(dir(nfl))
from playoffs import playoff_field

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'cache', 'clear_cache', 'config', 'downloader', 'get_current_season', 'get_current_week', 'load_combine', 'load_contracts', 'load_depth_charts', 'load_draft_picks', 'load_ff_opportunity', 'load_ff_playerids', 'load_ff_rankings', 'load_ffverse', 'load_ftn_charting', 'load_injuries', 'load_nextgen_stats', 'load_officials', 'load_participation', 'load_pbp', 'load_pfr_advstats', 'load_player_stats', 'load_players', 'load_rosters', 'load_rosters_weekly', 'load_schedules', 'load_snap_counts', 'load_stats', 'load_team_stats', 'load_teams', 'load_trades', 'utils_date', 'version']


In [9]:

sch = nfl.load_schedules([2023])
# convert to pandas if needed
if not isinstance(sch, pd.DataFrame):
    sch = sch.to_pandas()

print("SCHEDULE COLUMNS:")
print(list(sch.columns))
print("\nSCHEDULE SAMPLE:")
print(sch.head(3))


SCHEDULE COLUMNS:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']

SCHEDULE SAMPLE:
           game_id  season game_type  week     gameday   weekday gametime  \
0   2023_01_DET_KC    2023       REG     1  2023-09-07  Thursday    20:20   
1  2023_01_CAR_ATL    2023       REG     1  2023-09-10    Sunday    13:00   
2  2023_01_HOU_BAL    2023       REG     1  2023-09-10    Sunday    13:00   

  away_team  away_score home_team  ...  wind  away_qb_id  home_qb_id  \
0       DET

In [124]:
teams = nfl.load_teams()
if not isinstance(teams, pd.DataFrame):
    teams = teams.to_pandas()

print("TEAM COLUMNS:")
print(list(teams.columns))
print("\nTEAM SAMPLE:")
print(teams.head(40))


TEAM COLUMNS:
['team_abbr', 'team_name', 'team_id', 'team_nick', 'team_conf', 'team_division', 'team_color', 'team_color2', 'team_color3', 'team_color4', 'team_logo_wikipedia', 'team_logo_espn', 'team_wordmark', 'team_conference_logo', 'team_league_logo', 'team_logo_squared']

TEAM SAMPLE:
   team_abbr              team_name  team_id   team_nick team_conf  \
0        ARI      Arizona Cardinals     3800   Cardinals       NFC   
1        ATL        Atlanta Falcons      200     Falcons       NFC   
2        BAL       Baltimore Ravens      325      Ravens       AFC   
3        BUF          Buffalo Bills      610       Bills       AFC   
4        CAR      Carolina Panthers      750    Panthers       NFC   
5        CHI          Chicago Bears      810       Bears       NFC   
6        CIN     Cincinnati Bengals      920     Bengals       AFC   
7        CLE       Cleveland Browns     1050      Browns       AFC   
8        DAL         Dallas Cowboys     1200     Cowboys       NFC   
9        

In [4]:
def build_real_season_dfs(year: int):
    # Load in the schedules
    sch = nfl.load_schedules([year]).to_pandas()

    # Keep all regular season games
    sch = sch[(sch["season"] == year) & (sch["game_type"] == "REG")].copy()

    # Load in NFL Team Data, need to convert to pandas cause the nflreadpy is weird
    teams = nfl.load_teams().to_pandas()

    # Connect the NFL Team Data to the abbreviations in the schedule section
    team_cols = ["team_abbr", "team_name", "team_conf", "team_division"]

    # Do a left join for all home teams in the results
    sch = sch.merge(teams[team_cols],left_on="home_team",right_on="team_abbr").rename(columns={ "team_name": "Home",
        "team_conf": "Home_Conf", "team_division": "Home_Div"
        }).drop(columns=["team_abbr"])

    # Do a left join for all the away teams in the results
    sch = sch.merge(teams[team_cols],left_on="away_team",right_on="team_abbr").rename(columns={"team_name": "Away",
        "team_conf": "Away_Conf","team_division": "Away_Div"
        }).drop(columns=["team_abbr"])
  
    # Keep track of the margins for every game
    home_score = sch["home_score"]
    away_score = sch["away_score"]
    margin_home = home_score - away_score

    # Create an empty pandas series to store each game's winner and loser
    winner = pd.Series([None] * len(sch), index=sch.index)
    loser  = pd.Series([None] * len(sch), index=sch.index)
    tie = pd.Series([None] * len(sch), index=sch.index)

    # home wins if margin > 0, away wins if less than 0
    home_win = margin_home > 0
    away_win = margin_home < 0
    both_tie = margin_home == 0

    # Attach winners and losers based on the boolean function into the winner and lsoer series
    winner[home_win] = sch.loc[home_win, "Home"]
    loser[home_win]  = sch.loc[home_win, "Away"]  
    winner[away_win] = sch.loc[away_win, "Away"]
    loser[away_win]  = sch.loc[away_win, "Home"]
    tie_home = sch.loc[both_tie, "Home"]
    tie_away = sch.loc[both_tie, "Away"]
    # Create the results dataframe, exactly in the format I created my results in for the simulations
    results_df = pd.DataFrame({"Home": sch["Home"],"Away": sch["Away"],"Winner": winner,
        "Loser": loser, "Tie": tie, "Divisional": sch["Home_Div"] == sch["Away_Div"],
        "Conference": sch["Home_Conf"] == sch["Away_Conf"], "Margin": margin_home,
        "Home_Score": home_score, "Away_Score": away_score,
    })

    #group teams by amount of wins
    wins = results_df["Winner"].value_counts()
    losses = results_df["Loser"].value_counts()
    ties = pd.concat([tie_home, tie_away], ignore_index=True).value_counts()

  
    # map teams wins and losses to the standings
    standings = pd.DataFrame({"Team": teams["team_name"].unique()})
    standings["Wins"] = standings["Team"].map(wins).fillna(0).astype(int)
    standings["Losses"] = standings["Team"].map(losses).fillna(0).astype(int)
    standings["Ties"] = standings["Team"].map(ties).fillna(0).astype(int)

    div_wins = results_df.loc[results_df["Divisional"], "Winner"].value_counts()
    conf_wins = results_df.loc[results_df["Conference"], "Winner"].value_counts()
    div_losses = results_df.loc[results_df["Divisional"], "Loser"].value_counts()
    conf_losses = results_df.loc[results_df["Conference"], "Loser"].value_counts()    
    div_ties = results_df.loc[results_df["Conference"], "Tie"].value_counts()
    conf_ties = results_df.loc[results_df["Conference"], "Tie"].value_counts()
    # map teams divisional and conference wins
    standings["Div_Wins"] = standings["Team"].map(div_wins).fillna(0).astype(int)
    standings["Conf_Wins"] = standings["Team"].map(conf_wins).fillna(0).astype(int)
    standings["Div_Losses"] = standings["Team"].map(div_losses).fillna(0).astype(int)
    standings["Conf_Losses"] = standings["Team"].map(conf_losses).fillna(0).astype(int)
    standings["Div_Ties"] = standings["Team"].map(div_ties).fillna(0).astype(int)
    standings["Conf_Ties"] = standings["Team"].map(conf_ties).fillna(0).astype(int)

    # map all teams to correct conference and division
    team_info = teams.set_index("team_name")[["team_conf", "team_division"]]
    standings["Conference"] = standings["Team"].map(team_info["team_conf"].to_dict())
    standings["Division"] = standings["Team"].map(team_info["team_division"].to_dict())

    # cummulative margin for home and away teams grouped by team
    home_margin = (results_df["Home_Score"] - results_df["Away_Score"]).groupby(results_df["Home"]).sum()
    away_margin = (results_df["Away_Score"] - results_df["Home_Score"]).groupby(results_df["Away"]).sum()

    # merge them together
    total_margin = home_margin.add(away_margin, fill_value=0)
    standings["Total_Margin"] = standings["Team"].map(total_margin).fillna(0).astype(int)
   
    # remove teams with no wins/losses (ie, Oakland Raiders)
    standings = standings[~((standings["Wins"] == 0) & (standings["Losses"] == 0))]
 
    # Sort the standings by wins and total margin
    standings_df = standings.sort_values(["Wins", "Total_Margin"], ascending=[False, False]).reset_index(drop=True)
    return results_df, standings_df


In [5]:
total = Counter()

# Loop through the years and count the # of tiebreakers
for year in range(2002, 2026):
    results_df, standings_df = build_real_season_dfs(year)
    field, tbs = playoff_field(standings_df, results_df)
    total.update(tbs)
    print(year, "Playoff field")
    print(field)
    #print(standings_df)

print(total)
print(standings_df)
#print(results_df)

New England Patriots won a tiebreaker by division_record
New York Jets won a tiebreaker by common_games New England Patriots New York Jets
Philadelphia Eagles won a tiebreaker by conference_record
New England Patriots won a tiebreaker by division_record
Cleveland Browns won a tiebreaker by conference_record
New England Patriots won a tiebreaker by division_record
2002 Playoff field
   Conference  Seed                  Team   Division  Wins  Losses  Ties
0         AFC     1       Oakland Raiders   AFC West    11       5     0
1         AFC     2      Tennessee Titans  AFC South    11       5     0
2         AFC     3   Pittsburgh Steelers  AFC North    10       5     1
3         AFC     4         New York Jets   AFC East     9       7     0
4         AFC     5    Indianapolis Colts  AFC South    10       6     0
5         AFC     6      Cleveland Browns  AFC North     9       7     0
6         AFC     7  New England Patriots   AFC East     9       7     0
7         NFC     1   Philadelp